In [ ]:
project_path = "/home/jupyter"
import os
import sys

sys.path.append(project_path)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from google.cloud import bigquery

from fintrans_toolbox.src import bq_utils as bq
from fintrans_toolbox.src import table_utils as t


client = bigquery.Client()

In [ ]:
#Top 10 F2F Goods

Top10_F2F_spending_by_mccgoods = '''
WITH mcc_quarterly_spend AS (
  SELECT 
    SUM(spend) AS total_spend,
    time_period_value,
    mcc
  FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
  WHERE time_period_value IN (
    '2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4',
    '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4',
    '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1'
  )
  AND cardholder_origin_country = 'All' 
  AND cardholder_origin = 'UNITED KINGDOM' 
  AND merchant_channel = 'Face to Face'
  AND mcc IN (
    'ANTIQUE REPRODUCTION STORES',
'ANTIQUE SHOPS', 
        'ARTIST/CRAFT SHOPS',
        'ART DEALERS & GALLERIES',
        'AUTOMATED FUEL DISPENSERS',
        'AUTOMOTIVE PARTS STORES', 
        'AUTOMOTIVE TIRE STORES', 
        'BAKERIES', 
        'BICYCLE SHOPS/SALE/SERVICE',
        'BOAT DEALERS',
        'BOOKS/PERIODICALS/NEWSPAPERS',
        'BOOK STORES', 
        'CAMERA & PHOTO SUPPLY STORES',
        'CAMPER TRAILER DEALER',
        'CANDY/NUT/CONFECTION STORES',
        'CAR & TRUCK DEALERS/NEW/USED', 
        'CAR & TRUCK DEALERS/USED ONLY', 
        'CATALOG MERCHANT',
        'CHILDREN/INFANTS WEAR STORES',
        'CLOTHING/RENT/COSTUME/UNIFO',
        'COMBINATION CATALOG & RETAIL',
        'COMPUTERS/PERIPHERALS/SOFTWARE',
        'COSMETIC STORES', 
        'DAIRY PRODUCT STORES',
        'DEPARTMENT STORES', 
        'DIRECT SELL/DOOR-TO-DOOR',
        'DISCOUNT STORES',
        'DRAPERY & UPHOLSTERY STORES',
        'DUTY FREE STORES',
        'ELEC RAZOR STORES/SALE/SERV',
        'ELECTRONICS STORES', 
        'FABRIC STORES',
        'FAMILY CLOTHING STORES',
        'FIREPLACES & ACCESSORIES',
        'FLOOR COVERING STORES', 
        'FLORIST SUPPLIES/NURSERY STOCK',
        'FLORISTS',
        'FREEZER/MEAT LOCKERS',
        'FUEL DEALERS',
        'FURNITURE/EQUIP STORES',
        'FURRIERS AND FUR SHOPS',
        'GLASS/PAINT/WALLPAPER STORES', 
        'GIFT, CARD, NOVELTY STORES', 
        'GLASSWARE/CRYSTAL STORES',
        'GROCERY STORES/SUPERMARKETS',
        'HARDWARE STORES',
        'HOBBY, TOY & GAME STORES',
        'HOME SUPPLY WAREHOUSE STORES',
        'HOUSEHOLD APPLIANCE STORES', 
        'JEWELRY STORES', 
        'LUGGAGE/LEATHER STORES',
        'LUMBER/BUILD. SUPPLY STORES', 
        'MEN/BOYS CLOTHING/ACC STORES',
        'MENS/WOMENS CLOTHING STORES',
        'MISC APPAREL/ACCESS STORES',
        'MISC AUTO DEALERS - DEFAULT',
        'MISC FOOD STORES - DEFAULT',
        'MISC GENERAL MERCHANDISE', 
        'MISC HOME FURNISHING SPECIALTY', 
        'MISC SPECIALTY RETAIL', 
        'MOBILE HOME DEALERS', 
        'MOTOR HOME DEALERS',
        'MOTOR VEHICLE SUPPLY/NEW PARTS', 
        'MOTORCYCLE DEALERS', 
        'MUSIC STORES/PIANOS', 
        'NEWS DEALERS/NEWSSTANDS',
        'NURSURIES, LAWN/GARDEN SUPPLY',
        'OFFICE/PHOTO EQUIPMENT',
        'ONLINE MARKETPLACES',
        'OTHER DIRECT MARKETERS',
        'OUTBOUND TELEMARKETING MERCHNT',
        'PAINT, VARNISHES & SUPPLIES',
        'PET STORES/FOOD & SUPPLY',
        'PETROLEUM/PETROLEUM PRODUCTS',
        'PLUMBING/HEATING EQUIPMENT',
        'POSTAGE STAMPS',
        'PRECIOUS STONES/METALS/JEWELRY',
        'RECORD STORES',
        'RELIGIOUS GOODS STORES',
        'ROOFING/SIDING/SHEET METAL'
        'SERVICE STATIONS',
        'SHOE STORES',
        'SNOWMOBILE DEALERS',
        'SPORTING GOODS STORES',
        'SPORTS/RIDING APPAREL STORES',
        'STAMP & COIN STORES',
        'STATIONERY STORES',
        'STATIONERY/OFFICE SUPPLIES',
        'SWIMMING POOLS/SALES/SERV',
        'TENT AND AWNING SHOPS',
        'TELECOMMUNICATION EQUIPMENT',
        'UNIFORMS & COMMERCIAL CLOTHING',
        'USED MERCHANDISE STORES',
        'WIG AND TOUPEE STORES',
        'WOMENS READY TO WEAR STORES',
        'WRECKING SALVAGE YARDS',
'VIDEO AMUSEMENT GAME SUPPLY'
  )
  GROUP BY time_period_value, mcc
),
ranked_mcc AS (
  SELECT 
    *,
    RANK() OVER (PARTITION BY time_period_value ORDER BY total_spend DESC) AS mcc_rank
  FROM mcc_quarterly_spend
)
SELECT 
  time_period_value,
  mcc,
  total_spend
FROM ranked_mcc
WHERE mcc_rank <= 10
ORDER BY time_period_value, mcc_rank;
'''

# Run the query and load into a DataFrame
df_Top10_F2F_spending_by_mccgoods = client.query(Top10_F2F_spending_by_mccgoods).to_dataframe()

# Save to CSV
df_Top10_F2F_spending_by_mccgoods.to_csv('Top10_F2F_spending_by_mccgoods.csv', index=False)

print("Top 10 MCCs by quarter saved to 'Top10_F2F_spending_by_mccgoods.csv'")


In [ ]:
# Line Chart for 2019Q1 as base value = 100

import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV file
df = pd.read_csv("Top10_F2F_spending_by_mccgoods.csv")

# Pivot the data to have time_period_value as index and mcc as columns
pivot_df = df.pivot(index="time_period_value", columns="mcc", values="total_spend")

# Sort the index to ensure chronological order
pivot_df = pivot_df.sort_index()

# Normalize the data to 2019Q1 = 100
normalized_df = pivot_df.divide(pivot_df.loc["2019Q1"]).multiply(100)

# Filter for quarters from 2019Q1 to 2025Q1
quarters = pd.date_range(start="2019Q1", end="2025Q1", freq="Q").to_period("Q").astype(str)
normalized_df = normalized_df.loc[normalized_df.index.isin(quarters)]

# Plot the line chart
plt.figure(figsize=(12, 6))
for column in normalized_df.columns:
    plt.plot(normalized_df.index, normalized_df[column], label=column)

plt.title("Indexed Total Spending by MCC (2019Q1 = 100)")
plt.xlabel("Quarter")
plt.ylabel("Indexed Total Spend")
plt.xticks(rotation=45)
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.grid(True)
plt.show()


In [ ]:
#Traceable Graph

import pandas as pd
import plotly.express as px

# Load the CSV file
df = pd.read_csv("Top10_F2F_spending_by_mccgoods.csv")

# Pivot the data
pivot_df = df.pivot(index="time_period_value", columns="mcc", values="total_spend")
pivot_df = pivot_df.sort_index()

# Normalize to 2019Q1 = 100
normalized_df = pivot_df.divide(pivot_df.loc["2019Q1"]).multiply(100)

# Filter for 2019Q1 to 2025Q1
quarters = pd.date_range(start="2019Q1", end="2025Q1", freq="Q").to_period("Q").astype(str)
normalized_df = normalized_df.loc[normalized_df.index.isin(quarters)]

# Reshape for Plotly
normalized_df = normalized_df.reset_index().melt(id_vars="time_period_value", var_name="MCC", value_name="Indexed Spend")

# Create interactive chart
fig = px.line(normalized_df, x="time_period_value", y="Indexed Spend", color="MCC",
              title="Indexed Total Spending by MCC (2019Q1 = 100)",
              labels={"time_period_value": "Quarter"},
              markers=True)

# Adjust layout for better visibility
fig.update_layout(
    width=1200,
    height=600,
    margin=dict(l=80, r=350, t=60, b=60),
    xaxis_tickangle=-45,
    hovermode="x unified",
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.05,
        font=dict(size=10),
        traceorder="normal",
        itemsizing="constant"
    )
)

fig.show()


In [ ]:
#Line chart with base value 2019 ave

import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV file
df = pd.read_csv("Top10_F2F_spending_by_mccgoods.csv")

# Pivot the data to have time_period_value as index and mcc as columns
pivot_df = df.pivot(index="time_period_value", columns="mcc", values="total_spend")

# Sort the index to ensure chronological order
pivot_df = pivot_df.sort_index()

# Compute the average for 2019
quarters_2019 = [q for q in pivot_df.index if q.startswith("2019")]
average_2019 = pivot_df.loc[quarters_2019].mean()

# Normalize the data to 2019 average = 100
normalized_df = pivot_df.divide(average_2019).multiply(100)

# Filter for quarters from 2019Q1 to 2025Q1
quarters = pd.date_range(start="2019Q1", end="2025Q1", freq="Q").to_period("Q").astype(str)
normalized_df = normalized_df.loc[normalized_df.index.isin(quarters)]

# Plot the line chart
plt.figure(figsize=(12, 6))
for column in normalized_df.columns:
    plt.plot(normalized_df.index, normalized_df[column], label=column)

plt.title("Indexed Total Spending by MCC (2019 Average = 100)")
plt.xlabel("Quarter")
plt.ylabel("Indexed Total Spend")
plt.xticks(rotation=45)
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.grid(True)
plt.show()


In [ ]:
#Traceable

import pandas as pd
import plotly.express as px

# Load the CSV file
df = pd.read_csv("Top10_F2F_spending_by_mccgoods.csv")

# Pivot the data
pivot_df = df.pivot(index="time_period_value", columns="mcc", values="total_spend")
pivot_df = pivot_df.sort_index()

# Compute the average for 2019
quarters_2019 = [q for q in pivot_df.index if q.startswith("2019")]
average_2019 = pivot_df.loc[quarters_2019].mean()

# Normalize to 2019 average = 100
normalized_df = pivot_df.divide(average_2019).multiply(100)

# Filter for 2019Q1 to 2025Q1
quarters = pd.date_range(start="2019Q1", end="2025Q1", freq="Q").to_period("Q").astype(str)
normalized_df = normalized_df.loc[normalized_df.index.isin(quarters)]

# Reshape for Plotly
normalized_df = normalized_df.reset_index().melt(id_vars="time_period_value", var_name="MCC", value_name="Indexed Spend")

# Create interactive chart
fig = px.line(normalized_df, x="time_period_value", y="Indexed Spend", color="MCC",
              title="Indexed Total Spending by MCC (2019 Average = 100)",
              labels={"time_period_value": "Quarter"},
              markers=True)

# Adjust layout for better visibility
fig.update_layout(
    width=1200,
    height=600,
    margin=dict(l=80, r=350, t=60, b=60),
    xaxis_tickangle=-45,
    hovermode="x unified",
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.05,
        font=dict(size=10),
        traceorder="normal",
        itemsizing="constant"
    )
)

fig.show()


In [ ]:
#Top 10 Online Services

Top10_Online_spending_by_mccservices = '''
WITH mcc_quarterly_spend AS (
  SELECT 
    SUM(spend) AS total_spend,
    time_period_value,
    mcc
  FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel` 
  WHERE time_period_value IN (
    '2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4',
    '2021Q1', '2021Q2', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4',
    '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1'
  )
  AND cardholder_origin_country = 'All' 
  AND cardholder_origin = 'UNITED KINGDOM' 
  AND merchant_channel = 'Face to Face'
  AND mcc IN (
    'ACCOUNTANTS/AUDITORS/BOOKPR',
'ADVERTISING SERVICES',
'AQUARIUMS/SEAQUARIUMS',
'ARCHITECTURAL/ENG/SURVEY',
'AUTO BODY REPAIR SHOPS',
'AUTO PAINT SHOPS',
'AUTO SERVICE SHOPS/NON DEALER',
'BANDS/ORCHESTRAS/ENTERTAIN',
'BEAUTY/BARBER SHOPS',
'BILLIARD/POOL ESTABLISHMENT',
'BOWLING ALLEYS',
'BUS LINES',
'BUSINESS SERVICES - DEFAULT',
'BUSINESS/SECRETARIAL SCHOOL',
'CABLE, SAT, PAY TV/RADIO SVCS',
'CATERERS', 
'COLLEGES/UNIV/JC/PROFESSION',
'COMPUTER MAINT/SVCS - DEF',
'COMPUTER NETWORK/INFO SVCS',
'COMPUTER PROGRAM/SYS DESIGN',
'CONTRACTORS - CONCRETE',
'CORRESPONDENCE SCHOOLS',
'COURIER SERVICES',
'DANCE HALLS/STUDIOS/SCHOOLS',
'DETECTIVE/PROTECTIVE AGEN',
'DIGITAL GOODS APP(EXCL GAMES)',
'DIRECT SELL/DOOR-TO-DOOR',
'DRY CLEANERS',
'ELECTRICAL CONTRACTORS',
'EMPLOYMENT/TEMP HELP AGEN',
'EXTERMINATING/DISINFECT SERV',
'FURNITURE REUPHOLSTERY/REPAIR',
'GEN CONTRACTORS RESIDENTL/COML',
'HEALTH CARE',
'HEALTH & BEAUTY SPAS',
'INFORMATION RETRIEVAL SERVICES',
'INTRA-GOVERNMENT PURCHASES',
'LANDSCAPE/HORTICULTURAL SER',
'LAUNDRIES-FAMILY/COMMERCIAL',
'LAUNDRY/CLEANING/GARMENT SV',
'LOCAL COMMUTER TRANSPORT',
'MARINAS, SERVICE & SUPPLY',
'MEMBER CLUBS/SPORT/REC/GOLF',
'MISC PERSONAL SERV - DEF',
'MISC REPAIR SERVICES',
'MOTION PICTURE THEATRES',
'MOTOR FREIGHT CARRIERS',
'OUTBOUND TELEMARKETING MERCHNT',
'PARKING LOTS,METERS,GARAGES',
'PASSENGER RAILWAYS',
'PHOTO FINISH LABS/DEV',
'PROFESSIONAL SERVICES - DEF',
'PUBLIC GOLF COURSES',
'PUBLIC WAREHOUSING',
'RADIO/TV/STEREO REPAIR SHOP',
'RAILROADS',
'REAL EST AGNTS & MGRS RENTALS',
'RECREATION SERVICES',
'SHOE REPAIR/SHINE/HAT CLEAN',
'SMALL APPLIANCE REPAIR DEF',
'SPECIALTY CLEANING/POLISHING',
'SPORTING/RECREATIONAL CAMPS',
'STENOGRAPHIC SERVICES',
'TAX PAYMENTS',
'TAX PREPARATION SERVICE',
'TAXICABS/LIMOUSINES',
'TELECOMMUNICATION SERVICES',
'TELEGRAPH SERVICES',
'TESTING LABS (NON-MEDICAL)',
'THEATRICAL PRODUCERS',
'TIRE RETREAD/REPAIR SHOPS',
'TOLLS AND BRIDGE FEES',
'TOURIST ATTRACTIONS AND XHBT',
'TOWING SERVICES',
'TRADE/VOCATIONAL SCHOOLS',
'TRANSPORTATION SVCS - DEFAULT',
'TRAVEL AGENCIES',
'TYPESETTING/PLATE MAKING ETC',
'TYPEWRITER/SALES/SERVICE',
'VETERINARY SERVICES',
'WATCH/CLOCK/JEWELRY REPAIR',
'WELDING SERVICES',
       'VIDEO GAME ARCADES/ESTABLISH'
  )
  GROUP BY time_period_value, mcc
),
ranked_mcc AS (
  SELECT 
    *,
    RANK() OVER (PARTITION BY time_period_value ORDER BY total_spend DESC) AS mcc_rank
  FROM mcc_quarterly_spend
)
SELECT 
  time_period_value,
  mcc,
  total_spend
FROM ranked_mcc
WHERE mcc_rank <= 10
ORDER BY time_period_value, mcc_rank;
'''

# Run the query and load into a DataFrame
df_Top10_Online_spending_by_mccservices = client.query(Top10_Online_spending_by_mccservices).to_dataframe()

# Save to CSV
df_Top10_Online_spending_by_mccservices.to_csv('Top10_Online_spending_by_mccservices.csv', index=False)

print("Top 10 MCCs by quarter saved to 'Top10_Online_spending_by_mccservices'")

In [ ]:
# Line Chart for 2019Q1 as base value = 100

import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV file
df = pd.read_csv("Top10_Online_spending_by_mccservices.csv")

# Pivot the data to have time_period_value as index and mcc as columns
pivot_df = df.pivot(index="time_period_value", columns="mcc", values="total_spend")

# Sort the index to ensure chronological order
pivot_df = pivot_df.sort_index()

# Normalize the data to 2019Q1 = 100
normalized_df = pivot_df.divide(pivot_df.loc["2019Q1"]).multiply(100)

# Filter for quarters from 2019Q1 to 2025Q1
quarters = pd.date_range(start="2019Q1", end="2025Q1", freq="Q").to_period("Q").astype(str)
normalized_df = normalized_df.loc[normalized_df.index.isin(quarters)]

# Plot the line chart
plt.figure(figsize=(12, 6))
for column in normalized_df.columns:
    plt.plot(normalized_df.index, normalized_df[column], label=column)

plt.title("Indexed Total Spending by MCC (2019Q1 = 100)")
plt.xlabel("Quarter")
plt.ylabel("Indexed Total Spend")
plt.xticks(rotation=45)
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
#Traceable Graph

import pandas as pd
import plotly.express as px

# Load the CSV file
df = pd.read_csv("Top10_Online_spending_by_mccservices.csv")

# Pivot the data
pivot_df = df.pivot(index="time_period_value", columns="mcc", values="total_spend")
pivot_df = pivot_df.sort_index()

# Normalize to 2019Q1 = 100
normalized_df = pivot_df.divide(pivot_df.loc["2019Q1"]).multiply(100)

# Filter for 2019Q1 to 2025Q1
quarters = pd.date_range(start="2019Q1", end="2025Q1", freq="Q").to_period("Q").astype(str)
normalized_df = normalized_df.loc[normalized_df.index.isin(quarters)]

# Reshape for Plotly
normalized_df = normalized_df.reset_index().melt(id_vars="time_period_value", var_name="MCC", value_name="Indexed Spend")

# Create interactive chart
fig = px.line(normalized_df, x="time_period_value", y="Indexed Spend", color="MCC",
              title="Indexed Total Spending by MCC (2019Q1 = 100)",
              labels={"time_period_value": "Quarter"},
              markers=True)

# Adjust layout for larger chart and full legend
fig.update_layout(
    width=1200,
    height=600,
    margin=dict(l=80, r=350, t=60, b=60),  # extra space for legend
    xaxis_tickangle=-45,
    hovermode="x unified",
    legend=dict(
        orientation="v",
        yanchor="top",
        y=1,
        xanchor="left",
        x=1.05,
        font=dict(size=10),
        traceorder="normal",
        itemsizing="constant"
    )
)

fig.show()
